In [5]:
import pandas as pd
import ast
import os
import random
import M0  # Utilizing your provided M0.py file

# 1. Configuration and Directory Setup
# Ensure the output directory exists
OUT_DIR = "../data/output_data"
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

RESEARCHER_SKILLS_PATH = "../data/input_data/Set_4/large_researcher_skills.csv"
PROPOSAL_SKILLS_PATH   = "../data/input_data/Set_4/large_proposal_skills.csv"

# 2. Data Import 
# Load your uploaded data files
proposals_df = pd.read_csv(PROPOSAL_SKILLS_PATH) 
researchers_df = pd.read_csv(RESEARCHER_SKILLS_PATH) 

# Preprocess skills from string to actual sets/lists
proposals_df['skills'] = proposals_df['skills'].apply(lambda x: ast.literal_eval(x))
researchers_df['skills'] = researchers_df['skills'].apply(lambda x: ast.literal_eval(x))

# Create a dictionary of researcher skills for the metric function
researcher_skills_dict = dict(zip(researchers_df['researcher_name'], researchers_df['skills']))
all_names = researchers_df['researcher_name'].tolist()

def generate_single_teams():
    results = []
    
    for _, prop in proposals_df.iterrows():
        prop_link = prop['nsf_proposal_links_v0']
        prop_skills = prop['skills']
        
        # Pick one random 'target researcher' to lead the team
        lead_researcher = random.choice(all_names)
        
        # Use M0.py logic to create exactly 1 team for this lead researcher
        # This function handles lead inclusion and random sizing (1-4)
        teams_list = M0.create_teams_for_each_person(
            all_researchers=all_names, 
            target_researcher=lead_researcher, 
            num_of_teams=1
        )
        
        # Extract the single team generated
        team = teams_list[0]
        
        # Use M0.py logic to calculate the goodness score
        goodness = M0.apply_ultra_metric(prop_skills, team, researcher_skills_dict)
        
        results.append({
            'nsf_proposal_links_v0': prop_link,
            'lead_researcher': lead_researcher,
            'team': team,
            'goodness_score': goodness
        })
        
    return pd.DataFrame(results)

# 3. Execution
print("Generating 1 team per proposal using M0 logic...")
final_results = generate_single_teams()

# Save output
output_path = os.path.join(OUT_DIR, 'm0_teaming_results_single.csv')
final_results.to_csv(output_path, index=False)

print(f"Process complete. {len(final_results)} rows saved to {output_path}")

Generating 1 team per proposal using M0 logic...
Process complete. 500 rows saved to ../data/output_data/m0_teaming_results_single.csv
